# Kadane (Java)

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/fabiofurini/algorithms/blob/main/notebooks/ch18_recursive_vs_iterative/Kadane_java.ipynb)

Chapter 18 — Recursive versus iterative algorithms. Runnable notebook of `Kadane`: the algorithm in Java, run on the examples of the chapter, with the «Try it» cell to run it on your own data. The pseudo-code, the examples and the proofs are on the [chapter page](https://fabiofurini.github.io/algorithms/recursion/18-recursive-vs-iterative/).

The same notebook in [Python](https://colab.research.google.com/github/fabiofurini/algorithms/blob/main/notebooks/ch18_recursive_vs_iterative/Kadane_python.ipynb) and [C++](https://colab.research.google.com/github/fabiofurini/algorithms/blob/main/notebooks/ch18_recursive_vs_iterative/Kadane_cpp.ipynb); all the algorithms of the chapter in the [chapter notebook](https://colab.research.google.com/github/fabiofurini/algorithms/blob/main/notebooks/ch18_recursive_vs_iterative_java.ipynb).

**How to run it.** Click the «Open in Colab» badge, then *Runtime → Run all* (Ctrl+F9), or run one cell at a time with Shift+Enter, from top to bottom. The first cell checks that the JDK is there and installs it if it is missing. Each Java program is written to a file by a `%%writefile` cell, then compiled with `javac` and run with `java` in the next cell.

**Indices.** In the pseudo-code of the notes arrays are indexed from 1; in the code (as in all the languages used here) from 0: the element `a[i]` of the pseudo-code is the element of index `i-1` in the code, and a range `a[l..r]` becomes `a[l-1..r-1]`. The printed output of the notebooks uses indices from 0.

**Names.** The functions have the name of the pseudo-code, in the convention of the language (as in the introductory chapters): `FibRec` → `fibRec` (camelCase).

## Setup

On Colab the JDK is usually already there: the cell below checks it and installs it if it is missing.

In [1]:
!javac -version || (apt-get -qq update && apt-get -qq install -y default-jdk-headless)

javac 25.0.4.1


## Kadane: a better iterative algorithm

**Kadane**$(a,n)$ — *Input*: an array $a[1..n]$ of $n\ge1$ integers. *Output*: the maximum sum of a non-empty subarray $a[l..r]$. It is the compact version of the chapter: $cur\leftarrow\max(a[i],\,cur+a[i])$, $best\leftarrow\max(best,cur)$. The loop makes $n-1$ iterations: $\Theta(n)$ steps and $\Theta(1)$ memory, against the $n(n+1)/2$ pairs $(l,r)$ of brute force ($\Theta(n^2)$) and against the recursive divide and conquer ($\Theta(n\log_2 n)$): a good iterative algorithm can beat recursion.

On the example of the price changes of chapter 17 the result is $43$. Then, on pseudo-random arrays with values between $-50$ and $50$ (same generator in all languages, seed $n$), the iterations of `Kadane`, the pairs examined by brute force and the check that the two algorithms find the same maximum sum; finally the measured times.

In [2]:
%%writefile KadaneCompact.java
import java.util.*;

public class KadaneCompact {
    // kadane(a, n): maximum sum of a non-empty subarray of a[0..n-1]
    static int kadane(int[] a, int n) {
        int best = a[0];
        int cur = a[0];                 // maximum sum of a subarray ending at i
        int i = 1;
        while (i < n) {
            cur = Math.max(a[i], cur + a[i]);
            best = Math.max(best, cur);
            i = i + 1;
        }
        return best;
    }

    // Brute force: returns {maximum sum, pairs (l, r) examined}
    static long[] bruteForceBest(int[] a, int n) {
        int best = a[0];
        long pairs = 0;
        for (int l = 0; l < n; l++) {
            int s = 0;
            for (int r = l; r < n; r++) {
                s = s + a[r];
                pairs++;
                if (s > best) best = s;
            }
        }
        return new long[]{best, pairs};
    }

    // Integers between -50 and 50 (linear congruential generator)
    static int[] randomValues(int n, long seed) {
        long x = seed;
        int[] a = new int[n];
        for (int i = 0; i < n; i++) {
            x = (x * 1103515245L + 12345L) % 2147483648L;
            a[i] = (int) (x % 101L) - 50;
        }
        return a;
    }

    static double nowMs() {
        return System.nanoTime() / 1e6;
    }

    public static void main(String[] args) {
        int[] a = {13, -3, -25, 20, -3, -16, -23, 18, 20, -7, 12, -5, -22, 15, -4, 7};
        System.out.println("a = " + Arrays.toString(a));
        System.out.println("Kadane(a, 16) = " + kadane(a, 16));
        int[] b = {-3, -1, -2, -5};
        System.out.println("a = " + Arrays.toString(b) + "  Kadane(a, 4) = " + kadane(b, 4));
        System.out.println();
        System.out.printf("%6s %7s %11s %11s %9s  same sum?%n", "n", "Kadane", "BruteForce", "iterations", "pairs");
        double[][] times = new double[3][];
        int idx = 0;
        for (int n : new int[]{1000, 2000, 4000}) {
            int[] v = randomValues(n, n);
            double t0 = nowMs(); int k = kadane(v, n); double t1 = nowMs() - t0;
            t0 = nowMs(); long[] bf = bruteForceBest(v, n); double t2 = nowMs() - t0;
            System.out.printf("%6d %7d %11d %11d %9d  %s%n", n, k, bf[0], n - 1, bf[1], k == bf[0] ? "OK" : "ERROR");
            times[idx++] = new double[]{n, t1, t2};
        }
        System.out.println();
        System.out.println("Measured times (they depend on the machine):");
        for (double[] t : times) {
            System.out.printf("n=%-6d Kadane %9.4f ms   BruteForce %10.3f ms%n", (int) t[0], t[1], t[2]);
        }
    }
}

Writing KadaneCompact.java


In [3]:
!javac KadaneCompact.java && java KadaneCompact

a = [13, -3, -25, 20, -3, -16, -23, 18, 20, -7, 12, -5, -22, 15, -4, 7]
Kadane(a, 16) = 43
a = [-3, -1, -2, -5]  Kadane(a, 4) = -1

     n  Kadane  BruteForce  iterations     pairs  same sum?
  1000     547         547         999    500500  OK
  2000     981         981        1999   2001000  OK
  4000    1955        1955        3999   8002000  OK

Measured times (they depend on the machine):
n=1000   Kadane    0.2263 ms   BruteForce      4.169 ms
n=2000   Kadane    0.1672 ms   BruteForce      2.672 ms
n=4000   Kadane    0.2754 ms   BruteForce     16.264 ms


**Try it yourself.** Change $n$: the pairs of brute force grow as $n^2$, the iterations of Kadane as $n$.

In [4]:
%%writefile ProvaKadaneCompact.java
import java.util.*;

public class ProvaKadaneCompact {
    public static void main(String[] args) {
        // Try n = 8000: the pairs of brute force are n(n+1)/2 = 32 004 000, the iterations of Kadane 7999.
        int n = 8000;
        int[] v = KadaneCompact.randomValues(n, 12345);
        long[] bf = KadaneCompact.bruteForceBest(v, n);
        System.out.println(KadaneCompact.kadane(v, n) + " (" + bf[0] + ", " + bf[1] + ")");
    }
}

Writing ProvaKadaneCompact.java


In [5]:
!javac ProvaKadaneCompact.java && java ProvaKadaneCompact

1903 (1903, 32004000)


---

Teaching material by [Fabio Furini](https://fabiofurini.github.io/) — DIAG, Sapienza University of Rome.
Text, figures and data [CC BY 4.0](https://github.com/fabiofurini/algorithms/blob/main/LICENSE), code [MIT](https://github.com/fabiofurini/algorithms/blob/main/LICENSE-CODE).